In [16]:
import numpy as np
from collections import Counter


class TreeNode():
    def __init__(self, data, feature_idx, feature_val, prediction_probs) -> None:
        self.data = data
        self.feature_idx = feature_idx
        self.feature_val = feature_val
        self.prediction_probs = prediction_probs
        self.left = None
        self.right = None

    def node_def(self) -> str:

        unique_values, value_counts = np.unique(self.data[:,-1], return_counts=True)
        output = ", ".join([f"{value}->{count}" for value, count in zip(unique_values, value_counts)])            
        return f"LEAF | Label Counts = {output} | Pred Probs = {self.prediction_probs}"

class DecisionTreeFromScratch:
    """
    Decision Tree Classifier implemented from scratch
    For training, all possible percentiles of the feature values are used.
    The quality evaluation is done using entropy as a measure.
    """
    def __init__(self, max_depth, min_leaf) -> None:
        """
        Initialize DecisionTree with specified hyperparameters.
        max_depth - maximum depth of the tree.
        min_leaf - minimum number of data points required in a leaf node.
        """
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.tree = None
    
    def _entropy(self, class_probabilities) -> float:
        """Calculates entropy given a list of class probabilities."""
        return sum([-p * np.log2(p) for p in class_probabilities if p > 0])
    
    def _class_probabilities(self, labels) -> list:
        """Returns class probabilities based on label distribution."""
        total_count = len(labels)
        return [count / total_count for count in Counter(labels).values()]
    
    def _data_entropy(self, labels) -> float:
        """Calculates entropy for a given set of labels."""
        return self._entropy(self._class_probabilities(labels))
    
    def _partition_entropy(self, subsets) -> float:
        """Calculates weighted entropy for a given partition of data subsets."""
        total_count = sum(len(subset) for subset in subsets)
        return sum(self._data_entropy(subset) * (len(subset) / total_count) for subset in subsets)
    
    def _split(self, data, feature_index, feature_val) -> tuple:
        """Splits the data into two groups based on a threshold feature value."""
        mask = data[:, feature_index] < feature_val
        group1 = data[mask]
        group2 = data[~mask]
        return group1, group2
    
    def _find_best_split(self, data) -> tuple:
        """
        Finds the best feature and threshold to split data by calculating entropy.
        Returns the split with the lowest entropy.
        """
        min_part_entropy = float('inf')
        best_split = None
        
        n_features = data.shape[1] - 1  # Exclude label column
        for idx in range(n_features):
            feature_vals = np.percentile(data[:, idx], q=np.arange(1, 100, 5))
            for feature_val in feature_vals:
                g1, g2 = self._split(data, idx, feature_val)
                part_entropy = self._partition_entropy([g1[:, -1], g2[:, -1]])
                if part_entropy < min_part_entropy:
                    min_part_entropy = part_entropy
                    best_split = (g1, g2, idx, feature_val, min_part_entropy)
        
        return best_split if best_split else (None, None, None, None, None)
    
    def _find_label_probs(self, data) -> np.array:
        """Calculates label probabilities for a given dataset."""
        labels = data[:, -1].astype(int)
        total_labels = len(labels)
        label_probabilities = np.zeros(len(self.labels_in_train), dtype=float)
        for i, label in enumerate(self.labels_in_train):
            count = np.sum(labels == label)
            label_probabilities[i] = count / total_labels
        return label_probabilities
    
    def _create_tree(self, data, current_depth) -> TreeNode:
        """Recursively builds the decision tree."""
        if current_depth > self.max_depth:
            return None
        
        split = self._find_best_split(data)
        if not split:
            return None
        
        split1, split2, split_feature_idx, split_feature_val, _ = split
        label_probabilities = self._find_label_probs(data)
        
        node = TreeNode(data, split_feature_idx, split_feature_val, label_probabilities)
        
        # Check leaf node condition
        if len(split1) < self.min_leaf or len(split2) < self.min_leaf:
            return node
        
        current_depth += 1
        node.left = self._create_tree(split1, current_depth)
        node.right = self._create_tree(split2, current_depth)
        
        return node
    
    def _predict_one_sample(self, X) -> np.array:
        """Predicts the probability distribution for a single sample."""
        node = self.tree
        while node.left or node.right:
            if X[node.feature_idx] < node.feature_val:
                node = node.left
            else:
                node = node.right
        return node.prediction_probs
    
    def fit(self, X_train, Y_train) -> None:
        """Trains the decision tree model."""
        self.labels_in_train = np.unique(Y_train)
        train_data = np.concatenate((X_train, np.reshape(Y_train, (-1, 1))), axis=1)
        self.tree = self._create_tree(data=train_data, current_depth=0)
    
    def predict_proba(self, X_set) -> np.array:
        """Returns predicted probabilities for each sample in the dataset."""
        return np.apply_along_axis(self._predict_one_sample, 1, X_set)
    
    def predict(self, X_set) -> np.array:
        """Predicts the class labels for a given dataset."""
        pred_probs = self.predict_proba(X_set)
        return np.argmax(pred_probs, axis=1)
    
    def _print_recursive(self, node, level=0) -> None:
        """Recursively prints the tree structure."""
        if node:
            self._print_recursive(node.left, level + 1)
            print('    ' * level + f"-> Feature: {node.feature_idx}, Threshold: {node.feature_val}, Probabilities: {node.prediction_probs}")
            self._print_recursive(node.right, level + 1)
    
    def print_tree(self) -> None:
        """Prints the entire tree structure."""
        self._print_recursive(node=self.tree)


In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn import tree
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data



C:\Users\mihae\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [49]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_new_reduced, train_data_y, test_size=0.2, random_state=42)

# Print shapes to verify the split
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)

Training set shape: (8000, 50) (8000,)
Validation set shape: (2000, 50) (2000,)


In [57]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []
#for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
#    print(f"Training fold {fold + 1}")
tree = DecisionTreeFromScratch(max_depth=10, min_leaf=4)


# Train the model
tree.fit(X_train=X_train, Y_train=y_train)

# Make predictions
predictions = tree.predict(X_val)
predicted_probabilities = tree.predict_proba(X_val)

accuracy = accuracy_score(y_val, predictions)

# Output results
print("Predicted labels:", predictions)
print("Predicted probabilities:\n", predicted_probabilities)
print("Validation Accuracy:", accuracy)
# Optionally, print the tree structure
tree.print_tree()

Predicted labels: [3 1 2 ... 4 1 1]
Predicted probabilities:
 [[0.27586207 0.20689655 0.         0.37931034 0.13793103]
 [0.         1.         0.         0.         0.        ]
 [0.         0.00364964 0.91970803 0.         0.07664234]
 ...
 [0.         0.         0.375      0.         0.625     ]
 [0.         1.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.        ]]
Validation Accuracy: 0.7765
                                -> Feature: 19, Threshold: -234.05015936721, Probabilities: [0.         0.         0.01492537 0.98507463 0.        ]
                            -> Feature: 18, Threshold: 39.21919997004514, Probabilities: [0.06818182 0.00757576 0.00757576 0.90151515 0.01515152]
                                        -> Feature: 8, Threshold: -203.36135953397851, Probabilities: [0.         0.16666667 0.         0.5        0.33333333]
                                    -> Feature: 28, Threshold: 6.8015549588207485, Probabilities: [0. 